# Harmonizing two cohorts with BioMapper

This notebook is the worked demonstration of the harmonization protocol: resolve each cohort's
measured entities to a canonical KRAKEN knowledge-graph node, then link entities across cohorts
wherever they land on the same node.

It runs against a **keyless**, publicly reachable deployment. There is no API key to configure, no `.env` file
to create, and no credential of any kind in this notebook.

## The protocol in two steps

**Resolution** maps an entity name (or a local identifier) to a canonical KRAKEN node and returns
that node's full cross-namespace identifier set plus a provenance trace. It is deterministic for a
fixed graph state: no sampling step, no model call.

**Linking** harmonizes two cohorts. An entity from cohort A and an entity from cohort B are
equivalent when they resolve to the same canonical node. The test is an intersection of
*identifier* sets, never a string comparison of names.

One rule governs the whole thing and is easy to miss, so it comes early and is demonstrated
directly in Part 3: the linker **excludes** the structure-encoding namespaces (`INCHIKEY`, `INCHI`,
`SMILES`). Linking two entities through a shared structure hash would make the downstream
structural certificate circular, because the certificate would then be checking the very quantity
that formed the link.

## What this notebook covers

- Pinning the backend build, so every output below is attributable
- Resolving one entity end to end, with its per-annotator provenance trace
- Why the linker is identifier-only, shown on a real node
- Harmonizing two metabolite panels and reading the counts correctly
- One certified link and one refused link, and why a refusal is a result
- A false link that a newer graph build removed, and what that teaches about pair counts
- A second entity type (proteins), which behaves differently and needs a different caveat

---

## Part 0: Install and connect

Run every command below **from the repository root**, so that `.venv/bin/python` resolves.

```bash
uv venv .venv --python 3.11
uv pip install --python .venv/bin/python \
    'biomapper[benchmarks,notebook]==1.5.4' ipykernel nbconvert

# Register that environment under the kernel name this notebook DECLARES.
# This step is required, not optional. Neither Jupyter nor nbconvert runs the
# interpreter that launched it: both resolve the kernel named in the notebook's
# own `metadata.kernelspec`, which here is `biomapper-1-5-4`. Installing into
# `.venv` without registering it means the run either cannot find a kernel or
# quietly uses some other Python that may not have biomapper installed at all.
.venv/bin/python -m ipykernel install --user --name biomapper-1-5-4 \
    --display-name 'Python 3.11 (biomapper 1.5.4)'
```

Then either open the notebook in Jupyter and pick *Python 3.11 (biomapper 1.5.4)*, or reproduce the
saved outputs headlessly. This is the command that produced the outputs in this file:

```bash
.venv/bin/python -m nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=900 \
    notebooks/biomapper_tutorial.ipynb
```

If the kernel is not registered, that command fails with a `NoSuchKernel` error naming
`biomapper-1-5-4`. That loud failure is deliberate. The alternative, a notebook that declares a
generic `python3` kernel, runs against whatever interpreter happens to be on the path and then
prints this file's build-attribution cells as though they described it. A wrong pin that looks like
a pin is worse than no pin, so this notebook would rather not start than misreport which build
produced its numbers.

Three extras are used here. `notebook` supplies `nest_asyncio` (so `await` works inside a cell) and
`tqdm`. `benchmarks` supplies the certificate machinery used in Part 5 and the provenance reader
used immediately below; it also pulls `requests` and `rdkit`, which are heavier than a pure client
needs, so a project that only resolves and links can install plain `biomapper==1.5.4`. `nbconvert`
is needed only for the headless command and not for interactive use.

**On the API key.** The deployment this notebook targets takes no key. From 1.5.4 the client is
keyless by default: with no `api_key=` and no `BIOMAPPER_API_KEY` it sends no `X-API-Key` header,
and every entry point works, including the synchronous wrappers (`map_entity`, `map_entities`,
`map_dataset_file_sync`, `list_annotators` and friends). A configured key is sent unless
`anonymous=True` is passed, and a deployment that does require a key answers a keyless call with
`BioMapperAuthError` saying so.

The cells below still pass `anonymous=True`. It is no longer needed to go keyless; it is there so
that a reader who happens to have `BIOMAPPER_API_KEY` set sends nothing either, and reproduces the
saved outputs exactly. (Before 1.5.4 a missing key raised `BioMapperConfigError` and the
synchronous wrappers could not reach a keyless deployment at all; that gap is closed.)

In [1]:
import asyncio
import json
import os
import time

import httpx
import nest_asyncio

import biomapper
from biomapper import BioMapperClient, summarize
from biomapper.harmonize import curie_set, harmonize, predicted_curies

nest_asyncio.apply()

# The BioMapper resolution service. Keyless.
API_ENDPOINT = "https://biomapper.expertintheloop.io/api/v1"
# The Kestrel service in front of the KRAKEN graph. Keyless, and the authority on which build
# answered. Never send a key to this host.
KESTREL_URL = "https://kestrel.krakenkg.com/api"

API_TIMEOUT = 300.0

print(f"biomapper          {biomapper.__version__}")
print(f"BIOMAPPER_API_KEY  {'set' if os.getenv('BIOMAPPER_API_KEY') else 'not set'}")


def resolve(names, entity_type="biolink:SmallMolecule"):
    """Resolve a list of names through POST /map/batch, keylessly.

    One round trip for the whole list. `anonymous=True` forces keyless even if BIOMAPPER_API_KEY
    is set, so every reader's run matches the saved outputs; with no key configured, 1.5.4 is
    keyless anyway.
    """

    async def _run():
        async with BioMapperClient(
            anonymous=True, base_url=API_ENDPOINT, timeout=API_TIMEOUT
        ) as client:
            return await client.map_entities(
                [{"name": n} for n in names], entity_type=entity_type
            )

    return asyncio.run(_run())


def resolve_one(name):
    """Resolve ONE name through /map/entity, which returns the full response envelope.

    `map_entities` (the batch path) omits `raw_response`, and with it the per-annotator
    evidence, so the entity-level provenance trace needs this endpoint rather than the batch one.
    """

    async def _run():
        async with BioMapperClient(
            anonymous=True, base_url=API_ENDPOINT, timeout=API_TIMEOUT
        ) as client:
            return await client.map_entity(name)

    return asyncio.run(_run())

biomapper          1.5.4
BIOMAPPER_API_KEY  not set


---

## Part 1: Pin the build before reading any output

A resolution result is only interpretable against the graph that produced it. Node assignments
move between KRAKEN builds, and Part 6 is an entire section about one such move, so a notebook that
does not name its build is publishing unattributable numbers.

Everything below is read live from `GET {kestrel}/health`. Nothing here is hardcoded. Two version
fields are recorded rather than one, because they diverge at this build: `kg_version` is the graph
build and `kraken_package_version` is the package that built it. Recording either alone
under-describes the backend.

Do not read the service version from `/openapi.json`, whose `info.version` is a stale framework
default (`0.1.0`). The same is true of the resolution service's own `/health`, which reports
liveness rather than graph provenance, so both are printed below and labelled for what they are.

One honest limitation. The resolution service and the Kestrel service are pinned separately here,
because the resolution service does not report the graph build it is querying. The build printed
below is the one the Kestrel host reports, and it is the build the reference harmonization runs
recorded; if the resolution service were ever repointed at a different graph, this block would
describe a backend other than the one that answered. Recording both endpoints, as the cell does, is
what makes that checkable rather than invisible.

In [2]:
from biomapper.benchmarks.provenance import fetch_kg_build_info

kestrel_version, kg, health_error = fetch_kg_build_info(KESTREL_URL)

# `fetch_kg_build_info` is fail-soft: on a partial `/health` payload it substitutes the string
# "unknown" for a missing field and leaves `health_error` as None. Checking only `health_error`
# would therefore let a HALF-pinned run through and present its output as build-attributed, which
# is worse than no pin at all because "unknown" still looks like provenance. Check the fields.
required = {
    "kestrel_version": kestrel_version,
    "kg_version": kg.kg_version,
    "kraken_package_version": kg.kraken_package_version,
    "biolink_version": kg.biolink_version,
    "build_timestamp": kg.build_timestamp,
    "git_commit": kg.git_commit,
}
unpinned = sorted(k for k, v in required.items() if not v or v == "unknown")
if not kg.source_versions:
    unpinned.append("source_versions")
if health_error or unpinned:
    raise RuntimeError(
        "cannot pin the build, refusing to report unattributable output: "
        f"health_error={health_error!r} unpinned={unpinned}"
    )

print("=== Run provenance ===")
print(f"biomapper (client)      {biomapper.__version__}")
print(f"resolution endpoint     {API_ENDPOINT}  (keyless)")
print(f"kestrel endpoint        {KESTREL_URL}  (keyless)")
print(f"kestrel_version         {kestrel_version}")
print(f"kg_version              {kg.kg_version}")
print(f"kraken_package_version  {kg.kraken_package_version}")
print(f"biolink_version         {kg.biolink_version}")
print(f"build_timestamp         {kg.build_timestamp}")
print(f"git_commit              {kg.git_commit}")
print(f"ingested sources        {len(kg.sources)}")

print("\n=== Source versions (the circularity register) ===")
for name, version in sorted(kg.source_versions.items()):
    print(f"  {name:22s} {version}")

=== Run provenance ===
biomapper (client)      1.5.4
resolution endpoint     https://biomapper.expertintheloop.io/api/v1  (keyless)
kestrel endpoint        https://kestrel.krakenkg.com/api  (keyless)
kestrel_version         0.3.0
kg_version              2.3.0
kraken_package_version  2.2.0
biolink_version         4.2.5
build_timestamp         2026-09-25T00:34:42.623612+00:00
git_commit              3dd08a5b7b17e320ceb0c0091e09b781d4d90242
ingested sources        18

=== Source versions (the circularity register) ===
  babel                  2026jul22
  bio-age                2019
  bio-bmi                2023
  cdes                   accessed_2026-07-29
  ctkg                   4.1.7
  dakg                   0.5.7
  kg2                    2.10.2
  lipidmaps              accessed_2026-08-07
  loinc                  2.82
  long-covid             received_2026-09-17
  microbiome-kg          2.1.0
  multiomics-kg          2.1.0
  ncbigene               accessed_2026-08-31
  pgs-catalog     

The `sources` list doubles as a **circularity register**. Several reference vocabularies are
ingested into the graph being queried, so a benchmark whose gold answers come from one of them
measures coverage rather than independent accuracy. `lipidmaps`, `refmet`, `loinc` and `ncbigene`
are all in the list above. This matters again in Part 5: it is why the structural certificate is
built from a source *outside* the graph.

In [3]:
async def service_health():
    async with BioMapperClient(anonymous=True, base_url=API_ENDPOINT, timeout=60.0) as client:
        return await client.health_check()


# Liveness of the resolution service. Note that its `version` field is a framework default and is
# NOT the graph build; the graph build is the block printed above.
print(asyncio.run(service_health()))

{'status': 'healthy', 'version': '0.1.0', 'mapper_initialized': True}


---

## Part 2: Resolution, one entity end to end

`Creatinine` is a good first case because its node carries a large cross-namespace set including
both an InChIKey and a SMILES string, which Part 3 needs.

Three things come back and they are worth separating:

- the **canonical node** the resolver committed to (`chosen_kg_id`)
- the node's **cross-namespace identifier set** (`kg_equivalent_ids`), read off the graph
- a **certificate** describing what the graph asserts about that node and what independent
  evidence says about it

In [4]:
creatinine = resolve_one("Creatinine")

print(f"query              {creatinine.query_name}")
print(f"resolved           {creatinine.resolved}")
print(f"chosen_kg_id       {creatinine.chosen_kg_id}")
print(f"primary_curie      {creatinine.primary_curie}")
print(f"confidence         {creatinine.confidence_score} ({creatinine.confidence_tier})")
print(f"review flag        {creatinine.chosen_kg_id_review}")

print("\n=== Cross-namespace identifier set on the chosen node ===")
for prefix, ids in sorted(creatinine.kg_equivalent_ids.items()):
    print(f"  {prefix:22s} {', '.join(ids)}")

query              Creatinine
resolved           True
chosen_kg_id       RM:0028430
primary_curie      RM:0028430
confidence         5.152485699326107 (high)
review flag        None

=== Cross-namespace identifier set on the chosen node ===
  CAS                    200-466-7, 60-27-5, 82016-55-5
  CHEBI                  14029, 16737, 23406, 3910
  CHEMBL.COMPOUND        CHEMBL4849635, CHEMBL65567
  CHV                    0000003442
  DRUGBANK               DB11846
  HMDB                   HMDB0000562
  INCHIKEY               DDRJAANPRJIHGJ-UHFFFAOYSA-N
  KEGG                   C00791
  KEGG.COMPOUND          C00791
  LOINC                  LP14355-9, LP32035-5
  MESH                   D003404
  NCIT                   C399
  PUBCHEM.COMPOUND       137319715, 588
  REACT                  R-ALL-30733, R-ALL-446592
  RM                     0028430
  RXCUI                  2913
  RXNORM                 2913
  SMILES                 CN1CC(=O)NC1=N
  SNOMEDCT               15373003
  UMLS    

In [5]:
# The entity-level provenance trace: which annotator proposed which identifier, with its score
# and the rule that selected it. This is `assigned_ids` off the response envelope, which is why
# this section used /map/entity rather than the batch endpoint.
trace = creatinine.raw_response.result.assigned_ids

print("=== Provenance trace: per-annotator evidence for this commitment ===")
for annotator, namespaces in sorted(trace.items()):
    print(f"  {annotator}")
    for namespace, locals_ in sorted(namespaces.items()):
        for local_id, evidence in sorted(locals_.items()):
            detail = ", ".join(f"{k}={v}" for k, v in sorted(evidence.items())) or "(no score reported)"
            print(f"      {namespace}:{local_id}  {detail}")

=== Provenance trace: per-annotator evidence for this commitment ===
  kestrel-hybrid-search
      RM:0028430  resolved_via=canonical_preference, score=5.152485699326107
  metabolomics-workbench
      refmet_id:RM0028430  (no score reported)


Two annotators proposed the same identifier by different routes, and the trace records the score and
the selection rule (`resolved_via`) rather than only the outcome. That is what makes a resolution
auditable: a reader can see which evidence channel carried the commitment. Note that the RefMet
annotator reports its assignment without a score, so agreement here is agreement between a scored
and an unscored channel, not between two scores.

In [6]:
cert = creatinine.certificate

print("=== Resolution certificate (as returned by the API) ===")
print(f"state                      {cert.state}")
print(f"structure_status           {cert.structure_status}")
print(f"node_inchikey_blocks       {cert.node_inchikey_blocks}")
print(f"comparison_rule            {cert.comparison_rule}")
print(f"independent_source         {cert.independent_source}")
print(f"independent_inchikey_block {cert.independent_inchikey_block}")
print(f"independent_of_selection   {cert.independent_of_selection}")
print(f"lipid_resolution_level     {cert.lipid_resolution_level}")
print(f"refusal_reason             {cert.refusal_reason}")

=== Resolution certificate (as returned by the API) ===
state                      corroborated
structure_status           structure_present
node_inchikey_blocks       ['DDRJAANPRJIHGJ']
comparison_rule            inchikey_first_block_set_intersection/v1
independent_source         metabolomics-workbench
independent_inchikey_block DDRJAANPRJIHGJ
independent_of_selection   False
lipid_resolution_level     unavailable
refusal_reason             None


### Read that certificate carefully

The API certificate is useful but it is **not** a knowledge-graph-independent instrument. It reads
the chosen node's own InChIKey first and only falls back to an external source, and
`independent_of_selection` is the field that tells you whether the corroborating source was the
same registry that supplied the node. When that field is `False`, agreement is circular and carries
no information.

Part 5 uses a different and stricter instrument for exactly this reason. Keep the two apart:

| | API `ResolutionCertificate` | `certify_link` (Part 5) |
|---|---|---|
| Reads the linking node's own InChIKey | Yes, first | Never |
| Verdicts | corroborated, uncorroborated, contradicted, unavailable | certified, refuted, refused |
| Can certify from the graph node alone | Yes | No, it refuses |

One further caveat applies to both. An InChIKey first block is neither tautomer-invariant nor
charge-invariant, so a first-block comparison over-flags tautomers and salt forms while passing
stereoisomer differences whenever a side lacks its second block. A `refuted` or `contradicted`
verdict is a candidate for re-adjudication against an outside source, not a finding on its own.

---

## Part 3: The linker is identifier-only, and here is the proof

`biomapper.harmonize` builds one identifier set per entity and intersects them. Two functions make
the rule visible:

- `predicted_curies(...)` returns **everything** the resolver assigned: the chosen node plus every
  cross-reference on it.
- `curie_set(...)` is what the linker actually compares. It is `predicted_curies` **minus** every
  structure-encoding namespace.

The cell below prints the difference.

In [7]:
everything = predicted_curies(creatinine.chosen_kg_id, creatinine.kg_equivalent_ids)
linkable = curie_set(creatinine.chosen_kg_id, creatinine.kg_equivalent_ids)
excluded = sorted(everything - linkable)

print(f"assigned to the node        {len(everything)} CURIEs")
print(f"visible to the linker       {len(linkable)} CURIEs")
print(f"excluded by design          {len(excluded)}")
for curie in excluded:
    print(f"    {curie}")

assigned to the node        29 CURIEs
visible to the linker       27 CURIEs
excluded by design          2
    INCHIKEY:DDRJAANPRJIHGJ-UHFFFAOYSA-N
    SMILES:CN1CC(=O)NC1=N


### Why the exclusion is the load-bearing part of the protocol

Look at what was dropped: the node's own InChIKey, whose first block is `DDRJAANPRJIHGJ`.

Now suppose the linker had kept it. Two cohorts' entries for creatinine would link *because* they
share that structure hash. A certificate that then asked "do these two entities have the same
structure?" would be re-reading the quantity that formed the link. It would answer yes every time.
Precision would be 100 percent by construction and would mean nothing.

Excluding structure from the linker is what buys the certificate its independence. The link is made
on identifier evidence (`CHEBI`, `HMDB`, `KEGG`, `PUBCHEM`, `LOINC`, and so on); the structural
judgement is made afterwards, from a source outside the graph. Part 5 does exactly that, and Part 5
is only meaningful because of this cell.

One smaller normalization is visible in the identifier list too. The graph emits Biolink-style
database-section prefixes (`KEGG.COMPOUND`, `PUBCHEM.COMPOUND`) while cohort exports usually ship
the bare prefix (`KEGG`, `PUBCHEM`). Those are folded to one canonical form, so
`KEGG:C00791` and `KEGG.COMPOUND:C00791` compare equal. Genuinely different identifier spaces such
as `KEGG.GLYCAN` are deliberately left distinct.

---

## Part 4: Harmonizing two metabolite panels

The worked example is the first dataset pair from the milestone: UK Biobank Nightingale NMR against
Arivale Metabolon. Both sides below are short excerpts of the **assay panel metadata**, that is,
analyte names as each platform reports them. There is no participant-level data anywhere in this
notebook and no path into a gated dataset.

The excerpt is chosen to contain the interesting cases rather than the flattering ones:

- five analytes that both platforms measure and name differently
- the Nightingale lipoprotein-compartment phospholipid fields, which are the subject of Part 6
- the molecule-class field `Phosphatidylcholines` alongside two molecular phosphatidylcholine
  species from Arivale
- the bare `Linoleic acid` field next to its ratio field, which is the subject of Part 7
- two lipoprotein particle concentrations, which are not molecules and should not resolve

In [8]:
# UK Biobank Nightingale NMR field names (panel metadata, not measurements).
ukbb_nmr = [
    "Glucose",
    "Creatinine",
    "Citrate",
    "3-Hydroxybutyrate",
    "Phenylalanine",
    "Total cholesterol",
    "Phosphatidylcholines",
    "Phospholipids in small HDL",
    "Phospholipids in large LDL",
    "Total phospholipids in lipoprotein particles",
    "Linoleic acid",
    "Ratio of linoleic acid to total fatty acids",
    "Concentration of chylomicrons and extremely large VLDL particles",
    "Concentration of large LDL particles",
]

# Arivale Metabolon biochemical names (panel metadata, not measurements).
arivale_metabolon = [
    "glucose",
    "creatinine",
    "citrate",
    "3-hydroxybutyrate (BHBA)",
    "phenylalanine",
    "cholesterol",
    "phosphatidylcholine (16:0/22:5n3, 18:1/20:4)",
    "phosphatidylcholine (18:0/20:5, 16:0/22:5n6)",
    "linoleate (18:2n6)",
    "1-palmitoyl-2-oleoyl-GPC (16:0/18:1)",
]

print(f"UK Biobank Nightingale   {len(ukbb_nmr)} fields")
print(f"Arivale Metabolon        {len(arivale_metabolon)} biochemicals")

UK Biobank Nightingale   14 fields
Arivale Metabolon        10 biochemicals


In [9]:
ukbb_results = resolve(ukbb_nmr)
arivale_results = resolve(arivale_metabolon)
print(f"resolved {len(ukbb_results)} UK Biobank fields and {len(arivale_results)} Arivale biochemicals\n")

print(f"{'cohort':8s} {'canonical node':20s} {'CURIEs':>6s}  entity")
print("-" * 100)
for label, results in (("UKBB", ukbb_results), ("ARIVALE", arivale_results)):
    for r in results:
        n = len(curie_set(r.chosen_kg_id, r.kg_equivalent_ids))
        node = r.chosen_kg_id or "(none)"
        print(f"{label:8s} {node:20s} {n:6d}  {r.query_name}")

resolved 14 UK Biobank fields and 10 Arivale biochemicals

cohort   canonical node       CURIEs  entity
----------------------------------------------------------------------------------------------------
UKBB     RM:0135901               33  Glucose
UKBB     RM:0028430               27  Creatinine
UKBB     RM:0135884               63  Citrate
UKBB     RM:0153056                8  3-Hydroxybutyrate
UKBB     RM:0135897               44  Phenylalanine
UKBB     RM:0135639               60  Total cholesterol
UKBB     CHEBI:64482               5  Phosphatidylcholines
UKBB     HMDB:HMDB0302468         10  Phospholipids in small HDL
UKBB     HMDB:HMDB0302468         10  Phospholipids in large LDL
UKBB     CHEBI:13343              11  Total phospholipids in lipoprotein particles
UKBB     RM:0137483                2  Linoleic acid
UKBB     RM:0149976               35  Ratio of linoleic acid to total fatty acids
UKBB     (none)                    0  Concentration of chylomicrons and extremely la

In [10]:
report = harmonize(
    ukbb_results,
    arivale_results,
    a_label="ukbb_nightingale",
    b_label="arivale_metabolon",
)

print(json.dumps(report.summary(), indent=2))

{
  "n_links": 8,
  "ukbb_nightingale": {
    "total": 14,
    "comparable": 12,
    "unresolved": 2,
    "errors": 0,
    "linked": 7,
    "link_rate": 0.5833333333333334
  },
  "arivale_metabolon": {
    "total": 10,
    "comparable": 10,
    "unresolved": 0,
    "errors": 0,
    "linked": 8,
    "link_rate": 0.8
  }
}


In [11]:
print(f"{len(report.links)} harmonized pairs\n")
for link in report.links:
    shared = sorted(link.shared)
    head = ", ".join(shared[:4])
    more = f" (+{len(shared) - 4} more)" if len(shared) > 4 else ""
    print(f"{link.a_key}")
    print(f"    <-> {link.b_key}")
    print(f"    shared: {head}{more}")

8 harmonized pairs

3-Hydroxybutyrate
    <-> 3-hydroxybutyrate (BHBA)
    shared: CHEBI:20067, HMDB:HMDB0000357, KEGG:C01089, LIPIDBANK:DFA0271 (+4 more)
Citrate
    <-> citrate
    shared: ATC:A09AB04, ATC:B05CB02, CAS:106168-20-1, CAS:106809-03-4 (+59 more)
Creatinine
    <-> creatinine
    shared: CAS:200-466-7, CAS:60-27-5, CAS:82016-55-5, CHEBI:14029 (+23 more)
Glucose
    <-> glucose
    shared: ATC:B05CX01, ATC:V04CA02, ATC:V06DC01, CAS:200-075-1 (+29 more)
Phenylalanine
    <-> phenylalanine
    shared: CAS:136019-12-0, CAS:200-568-1, CAS:25191-15-5, CAS:63-91-2 (+40 more)
Phosphatidylcholines
    <-> phosphatidylcholine (16:0/22:5n3, 18:1/20:4)
    shared: CHEBI:64482, FMA:85434, LOINC:LP15796-3, PATHWHIZ.ELEMENTCOLLECTION:74 (+1 more)
Phosphatidylcholines
    <-> phosphatidylcholine (18:0/20:5, 16:0/22:5n6)
    shared: CHEBI:64482, FMA:85434, LOINC:LP15796-3, PATHWHIZ.ELEMENTCOLLECTION:74 (+1 more)
Ratio of linoleic acid to total fatty acids
    <-> linoleate (18:2n6)
    sh

### How to read those counts

Three numbers appear in the summary and they answer different questions. Conflating them is the
most common way to overstate a harmonization result.

**`comparable`** is how many entities resolved to at least one identifier, so they had the
opportunity to link. It is a measure of reach, not of correctness. Two of the UK Biobank fields
above are lipoprotein particle concentrations, and a particle count is not a molecule, so the
correct outcome is that they resolve to nothing at all. They appear in `unresolved`, not as
failures to link.

**`unresolved`** entities are refusal candidates. They are deliberately excluded from the
`link_rate` denominator: an entity that resolved to nothing never had an opportunity to link, so
scoring it as a miss would conflate non-resolution with non-equivalence. They are reported by name
rather than dropped.

**`n_links`** counts linked pairs, and linking is many-to-many by construction. Notice that
`Phosphatidylcholines` links to two different Arivale species, so one UK Biobank entity produces
two pairs. The pair count is therefore **not** a count of harmonized biological entities, and the
conservative figure is the distinct-entity count (`linked` per cohort) or a strict one-to-one
subset. Part 6 is what happens when this distinction is ignored.

In [12]:
print("unresolved on the UK Biobank side:")
for name in report.a_unresolved:
    print(f"    {name}")
print("\nunresolved on the Arivale side:")
for name in report.b_unresolved or ["(none)"]:
    print(f"    {name}")

print("\nerrored calls (a different claim from 'did not resolve'):")
print(f"    ukbb    {report.a_errors or '(none)'}")
print(f"    arivale {report.b_errors or '(none)'}")

fan_out = {}
for link in report.links:
    fan_out.setdefault(link.a_key, []).append(link.b_key)
print("\npairs per UK Biobank entity:")
for a_key, partners in sorted(fan_out.items(), key=lambda kv: -len(kv[1])):
    print(f"    {len(partners)}  {a_key}")

unresolved on the UK Biobank side:
    Concentration of chylomicrons and extremely large VLDL particles
    Concentration of large LDL particles

unresolved on the Arivale side:
    (none)

errored calls (a different claim from 'did not resolve'):
    ukbb    (none)
    arivale (none)

pairs per UK Biobank entity:
    2  Phosphatidylcholines
    1  3-Hydroxybutyrate
    1  Citrate
    1  Creatinine
    1  Glucose
    1  Phenylalanine
    1  Ratio of linoleic acid to total fatty acids


---

## Part 5: One certificate and one refusal

The link is identifier evidence. The certificate is a separate, structural judgement, and to be
worth anything it has to come from a source the graph did not supply.

`certify_link` compares two InChIKeys that were each resolved **independently of the node that
formed the link**. Here the independent source is the PubChem name index over PUG-REST, which is a
different service from the graph. `require_tags=True` makes the independence check fail closed: an
untagged side, or a side tagged `source="kg"`, is refused outright rather than quietly certified.

Its verdicts are `certified`, `refuted`, and `refused`. Three notes on the parameters and the
output before the cell:

- the `necs_*` parameter names carry a legacy cohort name from the benchmark this was ported from.
  They mean "side A" and "side B".
- the resolver is fail-soft, returning `None` on a network failure, so it also reports a status.
  `lookup_failed` is transient and must not be read as absence; `clean_miss` means the source
  genuinely has no structure for that name. Both are printed.
- PubChem throttles some HTTP clients, so an `httpx` client is passed in through the documented
  `session` hook.

In [13]:
from biomapper.benchmarks.scorers.independent_inchikey import PubChemInChIKeyResolver
from biomapper.benchmarks.scorers.link_certificate import certify_link

# The resolver accepts any object exposing `.get(url, timeout=...)` -> `.status_code` / `.text`.
# httpx is already a core dependency and is not throttled by PubChem the way `requests` is here.
pubchem_http = httpx.Client(follow_redirects=True, timeout=30.0)
structures = PubChemInChIKeyResolver(session=pubchem_http)

# Both sides are resolved by the SAME external service, the PubChem name index. The independence
# that matters is from the KG node that formed the link, not from each other, so the tag is the
# same on both sides and is set whether or not the lookup found anything: it records where the
# structure was sought, which is what the fail-closed provenance check needs.
SOURCE_TAG = "pubchem-name"


def independent_block(name):
    """First InChIKey block for `name` from PubChem, with its status. Never reads the KG node."""
    block, status = structures.block_for_name_status(name)
    time.sleep(0.25)  # PUG-REST courtesy pacing
    return block, status


def certify(a_name, b_name):
    a_block, a_status = independent_block(a_name)
    b_block, b_status = independent_block(b_name)
    result = certify_link(
        a_block,
        b_block,
        necs_source=SOURCE_TAG,
        cohort_source=SOURCE_TAG,
        require_tags=True,
    )
    print(f"link             {a_name!r}  <->  {b_name!r}")
    print(f"  side A         block={a_block}  status={a_status}")
    print(f"  side B         block={b_block}  status={b_status}")
    print(f"  verdict        {result.verdict}")
    print(f"  stereo_checked {result.stereo_checked}")
    print(f"  reason         {result.reason}")
    return result


try:
    print("### A certified link\n")
    certified = certify("Glucose", "glucose")
    print("\n### A refused link\n")
    refused = certify("Phosphatidylcholines", "phosphatidylcholine (16:0/22:5n3, 18:1/20:4)")
finally:
    pubchem_http.close()

### A certified link



link             'Glucose'  <->  'glucose'
  side A         block=WQZGKKKJIJFFOK  status=success
  side B         block=WQZGKKKJIJFFOK  status=success
  verdict        certified
  stereo_checked False
  reason         independent structures agree at connectivity (stereo not resolvable)

### A refused link



link             'Phosphatidylcholines'  <->  'phosphatidylcholine (16:0/22:5n3, 18:1/20:4)'
  side A         block=DRHYTKZTKWDSBJ  status=success
  side B         block=None  status=clean_miss
  verdict        refused
  stereo_checked False
  reason         no independent structure on the cohort side — counts-only, not certified off the KG


### The certified link

Both sides resolved to the same connectivity block from PubChem, and the graph node that formed the
link (an RM node) supplied neither of them. That agreement is informative precisely because of
Part 3: if the linker had been allowed to use the node's InChIKey, this comparison would have been
a tautology.

Note `stereo_checked` is `False`. The PubChem name index returns connectivity at the granularity
used here, so the comparison ran at block 1 only and the certificate says so rather than implying a
stereo layer was checked. A connectivity-only pass is weaker evidence than a full-key pass, and
labelling it is the difference between a certificate and a claim.

**What this example does not show.** `Glucose` and `glucose` differ only in capitalization, so this
pair demonstrates the mechanics of `certify_link` and the independence property, not the harder case
of two cohorts naming the same compound differently. That harder case cannot be certified from names
at all on this panel: its genuinely differently-named linked pairs, `3-hydroxybutyrate (BHBA)` and
`linoleate (18:2n6)`, carry vendor decoration that the PubChem name index does not match, so both
return a clean miss and would be refused for want of an independent structure rather than certified.
Certifying those needs the cohort's own published cross-reference identifiers (an HMDB accession or
a PubChem CID) as the independent anchor, which is what `block_for_provided` on the same resolver is
for, and which a name-only arm by construction does not have. This is one concrete reason the
provided-identifier arm is reported separately rather than blended in: it nearly doubles the
certifications that are independent of the registry that supplied the node, even where it does not
add a single harmonized molecule.

### The refused link

The Arivale side is a molecular phosphatidylcholine species written in shorthand. PubChem's name
index has no structure for that string, so there is no independent structure on that side and the
certificate **refuses**. It does not fall back to the graph node.

**A refusal is a first-class result, not a failure.** It is the protocol declining to assert
something it cannot support, and for this entity class it is the *correct* outcome rather than a
gap to be closed. Species-level and sum-composition lipids have no unique structure to certify
against: a shorthand name names a composition, and several distinct molecules satisfy it. Those
entities are harmonized at the RefMet or Goslin canonical level and reported as refused, which is
consistent with field-standard lipid identification. In the full milestone run, refusal is the
majority outcome on the Arivale metabolite panel, and reading that as an error rate would be a
straightforward misreading of the method.

The three verdicts are not one good and two bad. `certified` means independent structures agree.
`refuted` means they disagree, which flags a candidate for human adjudication rather than proving
the resolver wrong. `refused` means the question could not be asked independently, which is honest
rather than negative.

---

## Part 6: A false link, and why pair counts are not coverage

This is the most useful thing in the notebook, because it is a real correction rather than an
illustration.

Under KRAKEN 2.1.1, **seventeen** Nightingale lipoprotein-compartment phospholipid fields all
resolved to the same node, `CHEBI:64482`, which is the *generic phospholipid class*. Two Arivale
entries, the molecular species `phosphatidylcholine (16:0/22:5n3, 18:1/20:4)` and
`phosphatidylcholine (18:0/20:5, 16:0/22:5n6)`, resolved there too. Every compartment field
therefore linked to both species, producing 17 x 2 = **34 pairs**. The seventeen fields were the
whole Nightingale phospholipid family: total phospholipids in lipoprotein particles, and
phospholipids in VLDL, LDL, HDL, IDL, chylomicrons and extremely large VLDL, very large VLDL, large
VLDL, medium VLDL, small VLDL, very small VLDL, large LDL, small LDL, very large HDL, large HDL,
medium HDL, and small HDL.

Those 34 pairs were not cross-cohort equivalences. A compartment-specific measurement of how much
phospholipid is carried in small HDL is not the same entity as a named phosphatidylcholine species.

Under the current build the compartment fields move off the class node while the two Arivale
species stay on it, so the spurious links disappear and one class-level link survives. The cell
below shows that directly against the live graph.

In [14]:
by_name = {r.query_name: r for r in ukbb_results}
arivale_by_name = {r.query_name: r for r in arivale_results}

compartment_fields = [
    "Phospholipids in small HDL",
    "Phospholipids in large LDL",
    "Total phospholipids in lipoprotein particles",
]
class_field = "Phosphatidylcholines"
pc_species = [
    "phosphatidylcholine (16:0/22:5n3, 18:1/20:4)",
    "phosphatidylcholine (18:0/20:5, 16:0/22:5n6)",
]

print("Arivale molecular species, current node:")
for name in pc_species:
    print(f"    {arivale_by_name[name].chosen_kg_id:16s} {name}")

print(f"\nUK Biobank molecule-class field:")
print(f"    {by_name[class_field].chosen_kg_id:16s} {class_field}")

print("\nUK Biobank compartment-level fields:")
for name in compartment_fields:
    print(f"    {by_name[name].chosen_kg_id:16s} {name}")

linked_pairs = {(lk.a_key, lk.b_key) for lk in report.links}
print("\nDo the compartment fields link to the molecular species?")
for field in compartment_fields:
    for species in pc_species:
        print(f"    {(field, species) in linked_pairs}   {field}  <->  {species}")

print("\nDoes the class field link to them?")
for species in pc_species:
    print(f"    {(class_field, species) in linked_pairs}   {class_field}  <->  {species}")

Arivale molecular species, current node:
    CHEBI:64482      phosphatidylcholine (16:0/22:5n3, 18:1/20:4)
    CHEBI:64482      phosphatidylcholine (18:0/20:5, 16:0/22:5n6)

UK Biobank molecule-class field:
    CHEBI:64482      Phosphatidylcholines

UK Biobank compartment-level fields:
    HMDB:HMDB0302468 Phospholipids in small HDL
    HMDB:HMDB0302468 Phospholipids in large LDL
    CHEBI:13343      Total phospholipids in lipoprotein particles

Do the compartment fields link to the molecular species?
    False   Phospholipids in small HDL  <->  phosphatidylcholine (16:0/22:5n3, 18:1/20:4)
    False   Phospholipids in small HDL  <->  phosphatidylcholine (18:0/20:5, 16:0/22:5n6)
    False   Phospholipids in large LDL  <->  phosphatidylcholine (16:0/22:5n3, 18:1/20:4)
    False   Phospholipids in large LDL  <->  phosphatidylcholine (18:0/20:5, 16:0/22:5n6)
    False   Total phospholipids in lipoprotein particles  <->  phosphatidylcholine (16:0/22:5n3, 18:1/20:4)
    False   Total phospho

### What moved, and what that means

The compartment fields now sit on `HMDB:HMDB0302468` and `CHEBI:13343`, off the generic class node,
while the two Arivale species remain on `CHEBI:64482`. The two species went from linking eighteen
UK Biobank fields each to one each. Seventeen compartment-level links are dropped and one
class-level link is kept.

**Be precise about what that surviving link is.** It joins a molecule-*class* field on one side to a
*named molecular species* on the other, and the two meet on the generic class node rather than on a
shared molecular identity. It is retained as a **counts-only** link, not as a verified molecular
equivalence. It is in fact the very link whose certificate Part 5 refuses, because the species side
has no independent structure to check against. Keeping it is reasonable in the narrow sense that a
class field and a member of that class are not an obvious error, which is more than could be said
for the seventeen compartment links, but it has not been validated and must not be reported as a
certified equivalence.

Across the full panel this took the metabolite arm from 63 harmonized pairs to 28 in the name-only
channel, and from 131 to 96 with cohort identifiers supplied. **Thirty-five pairs were lost and none
were gained**, and thirty-four of those thirty-five are the fan-out above correcting itself.

Three readings follow, and the third is the one that generalizes.

**This is a gain in fidelity, not a loss of coverage.** The equivalence asserted under the older
build was never a real cross-cohort equivalence. Nothing correct was removed.

**The Arivale side did not move at all.** Its counts are identical before and after, which is the
signature of a one-sided artifact rather than a change in method. If a build change moves both
sides, something else is going on.

**A pair count can rise while precision falls.** Because linking is many-to-many, a single entity
resolving to an over-general node multiplies pairs across everything else that lands there. The
number goes up and the result gets worse. This is why the conservative figures are the
distinct-entity counts and the strict one-to-one subset, and why a harmonization result reported as
a single pair count is not interpretable.

---

## Part 7: A regression in the other direction

The thirty-fifth lost pair is not a correction, and reporting only the flattering half of a build
change would be the same failure in reverse.

`Linoleic acid` moved to a different RefMet node, while Arivale's `linoleate (18:2n6)` and the UK
Biobank field `Ratio of linoleic acid to total fatty acids` both stayed where they were. So the
bare analyte field lost its link and the *ratio* field kept it. A molecule-level equivalence was
traded for a ratio-level one, which is a worse link that happens to preserve the count.

In [15]:
for name in ["Linoleic acid", "Ratio of linoleic acid to total fatty acids"]:
    print(f"{by_name[name].chosen_kg_id:14s} {name}")
print(f"{arivale_by_name['linoleate (18:2n6)'].chosen_kg_id:14s} linoleate (18:2n6)")

print("\nlinks involving linoleic acid:")
for link in report.links:
    if "inole" in link.a_key or "inolea" in link.b_key:
        print(f"    {link.a_key}  <->  {link.b_key}")

RM:0137483     Linoleic acid
RM:0149976     Ratio of linoleic acid to total fatty acids
RM:0149976     linoleate (18:2n6)

links involving linoleic acid:
    Ratio of linoleic acid to total fatty acids  <->  linoleate (18:2n6)


A ratio is not a molecule. The link is technically present and biologically misleading, which is
exactly the kind of case that a strict one-to-one count surfaces and a raw pair count hides.

---

## Part 8: A second entity type

The same resolve-then-link protocol runs unchanged on a different entity type. Only the Biolink
category changes.

The panel below pairs UK Biobank Olink assay names against Arivale proteomics assay names, and it
is deliberately loaded with the informative cases: HGNC symbol renames, where the two platforms
name the same protein differently.

In [16]:
ukbb_olink = ["ADGRE2", "CCN1", "CCN3", "ADGRG1", "IL6", "TNF", "CD6", "LEP", "PCSK9"]
arivale_proteomics = ["EMR2", "CYR61", "NOV", "GPR56", "IL6", "TNF", "CD6", "Leptin", "PCSK9"]

ukbb_prot = resolve(ukbb_olink, entity_type="biolink:Protein")
arivale_prot = resolve(arivale_proteomics, entity_type="biolink:Protein")

protein_report = harmonize(
    ukbb_prot, arivale_prot, a_label="ukbb_olink", b_label="arivale_proteomics"
)
print(json.dumps(protein_report.summary(), indent=2))

print(f"\n{'ukbb assay':12s} {'arivale assay':14s} {'node':14s} shared UniProt accessions")
print("-" * 92)
nodes = {r.query_name: r.chosen_kg_id for r in ukbb_prot}
for link in protein_report.links:
    accessions = sorted(c for c in link.shared if c.startswith("UNIPROTKB:"))
    shown = ", ".join(a.split(":", 1)[1] for a in accessions[:3])
    more = f" (+{len(accessions) - 3})" if len(accessions) > 3 else ""
    print(f"{link.a_key:12s} {link.b_key:14s} {nodes[link.a_key]:14s} {shown}{more}")

{
  "n_links": 9,
  "ukbb_olink": {
    "total": 9,
    "comparable": 9,
    "unresolved": 0,
    "errors": 0,
    "linked": 9,
    "link_rate": 1.0
  },
  "arivale_proteomics": {
    "total": 9,
    "comparable": 9,
    "unresolved": 0,
    "errors": 0,
    "linked": 9,
    "link_rate": 1.0
  }
}

ukbb assay   arivale assay  node           shared UniProt accessions
--------------------------------------------------------------------------------------------
ADGRE2       EMR2           HGNC:3337      A0ACM8QGC0, A0JNV7, B4DWB8 (+1)
ADGRG1       GPR56          HGNC:4512      A0A024R6U7, A0A0S2Z517, B3KQN7 (+3)
CCN1         CYR61          HGNC:2654      O00622, Q6FI18
CCN3         NOV            HGNC:7885      P48745
CD6          CD6            HGNC:1691      P30203, Q6AZ88, Q8N4Q7
IL6          IL6            HGNC:6018      B4DNQ5, B4DVM1, B5MC21 (+3)
LEP          Leptin         HGNC:6553      A4D0Y8, P41159
PCSK9        PCSK9          HGNC:20001     A0A669KBG0, A0AAQ5BGX4, Q8NBP7
TNF    

### Two caveats, because 9 of 9 is not an accuracy claim

**This panel was selected to contain the renames.** `ADGRE2` and `EMR2`, `ADGRG1` and `GPR56`,
`CCN1` and `CYR61`, `CCN3` and `NOV`, `LEP` and `Leptin` are all the same protein under a current
and a superseded label, and the resolver reconciles them through the shared node. That is the
capability worth demonstrating. It is not a sample from which any rate should be read: on the full
panels, the large majority of protein links join assays that already use the *identical* gene
symbol, so a name-only protein link rate is much closer to string identity than the metabolite case
is, where the two vendors name the same compound differently. The informative tail is the renamed
symbols, which is what this excerpt is made of.

**The anchor is the accession, and it arrives through the node.** The canonical node here is an
HGNC node; the UniProt accessions come from the node's cross-namespace set and are what the
identifier intersection actually matches on. Both platforms publish accessions, which is why the
protein arm is the one symmetric case in the milestone. Note also that several accessions are
shared per link, including isoform and variant entries, so an accession count is not a protein
count either.

Genes are deliberately not demonstrated here. Gene symbols appear in both cohorts only as
annotation attached to the proteomic assays, so a gene arm would restate the protein arm under a
different label, and gene resolution accuracy has to be reported per target namespace rather than
as a single cross-namespace figure. Those numbers are under active diagnosis and are not quoted in
this notebook.

---

## Part 9: Resolution reach is not correctness

One number deserves its own section because it is the easiest figure in the whole protocol to
misread. A high resolution rate means the resolver returned a node. It does not mean the node is
right, and it does not mean the input was the kind of thing that *should* have a node.

The cell below submits three inputs that are not metabolites: a patent-derived catalog identifier,
a chemical supplier catalog code, and a lipid shorthand fragment.

In [17]:
not_metabolites = ["SCHEMBL20255602", "EN300-1697854", "Val-C20:0"]
junk = resolve(not_metabolites)

for r in junk:
    node = r.chosen_kg_id or "(none)"
    score = f"{r.confidence_score:.2f}" if r.confidence_score is not None else "n/a"
    print(f"{r.query_name:18s} resolved={r.resolved!s:5s} node={node:24s} score={score} tier={r.confidence_tier}")

s = summarize(junk)
print(f"\nresolution rate on inputs that should have no node: {s.resolution_rate:.0%}")

SCHEMBL20255602    resolved=True  node=CHEBI:181740             score=2.51 tier=high
EN300-1697854      resolved=True  node=HMDB:HMDB0254796         score=0.78 tier=low
Val-C20:0          resolved=True  node=RM:0137828               score=0.85 tier=low

resolution rate on inputs that should have no node: 100%


Everything resolved. That is the point. With the default annotator set, fuzzy matching returns
*some* node for almost any string, so `resolved` is close to useless as a quality filter on its own.

Confidence tier helps but does not rescue it. Two of the three above land in the `low` tier, which a
tier filter would catch, but the patent-derived identifier comes back `high`, so tier filtering
alone still admits a confident match to an unrelated entity. The reliable control is to **restrict
the annotator set**: asking only the Metabolomics Workbench annotator means a string that is not in
RefMet comes back genuinely unresolved rather than fuzzily matched.

This is the same lesson as Part 4's `comparable` count and Part 6's pair count, arriving from a
third direction: every denominator in this protocol has to be read for what it actually measures.

In [18]:
async def refmet_run():
    async with BioMapperClient(anonymous=True, base_url=API_ENDPOINT, timeout=API_TIMEOUT) as client:
        return await client.map_entities(
            [{"name": n} for n in not_metabolites],
            annotators=["metabolomics-workbench"],
        )


refmet_only = asyncio.run(refmet_run())
for r in refmet_only:
    node = r.chosen_kg_id or "(none)"
    print(f"{r.query_name:18s} resolved={r.resolved!s:5s} node={node}")

SCHEMBL20255602    resolved=False node=(none)
EN300-1697854      resolved=False node=(none)
Val-C20:0          resolved=False node=(none)


---

## Part 10: File-based input for a real panel

The excerpts above are Python lists so the notebook stays readable. A real cohort panel is a file,
and handing the server the file streams per-row results back over `POST /map/dataset/stream`
instead of materializing a list.

`provided_id_columns` is the channel for a cohort's own published identifiers, and it is worth
knowing that it is a genuinely separate arm rather than a free improvement. Supplying identifiers
is worth several points on name-input benchmarks, so a figure that blends the two channels is not
comparable to a published name-input number. In the milestone's clinical-lab arm, supplying LOINC
codes on one side only actually *reduced* harmonization, because it pulled that cohort into an
identifier-specific region of the graph that the name-input cohort never reaches. Keep the two arms
apart and report both.

In [19]:
from pathlib import Path

fixture = Path("../tests/fixtures/metabolites_sample.tsv")
print(fixture.read_text().strip())


async def stream_dataset():
    rows = []
    async with BioMapperClient(anonymous=True, base_url=API_ENDPOINT, timeout=API_TIMEOUT) as client:
        async for r in client.map_dataset_file_iter(
            fixture,
            name_column="name",
            provided_id_columns=["hmdb_id"],
        ):
            rows.append(r)
    return rows


streamed = asyncio.run(stream_dataset())
print(f"\n{len(streamed)} rows streamed back\n")
for r in streamed:
    print(f"  {r.query_name:16s} -> {r.primary_curie}")

name	hmdb_id
L-Histidine	HMDB00177
Glucose	HMDB00122
Pyruvic acid	HMDB00243
L-Alanine	HMDB00161
Citric acid	HMDB00094
L-Tyrosine	HMDB00158
Lactic acid	HMDB00190
L-Glutamic acid	HMDB00148
Glycine	HMDB00123
L-Phenylalanine	HMDB00159



10 rows streamed back

  L-Histidine      -> HMDB:HMDB0000177
  Glucose          -> HMDB:HMDB0000122
  Pyruvic acid     -> HMDB:HMDB0000243
  L-Alanine        -> HMDB:HMDB0000161
  Citric acid      -> HMDB:HMDB0000094
  L-Tyrosine       -> HMDB:HMDB0000158
  Lactic acid      -> HMDB:HMDB0000190
  L-Glutamic acid  -> HMDB:HMDB0000148
  Glycine          -> HMDB:HMDB0000123
  L-Phenylalanine  -> HMDB:HMDB0000159


Two things to know about this path. `confidence_score` is always `None` for dataset-stream results,
because the stream emits a slimmer per-row payload than `/map/batch` and carries no annotator
scores; use the batch path when you need tiers. And if you use the synchronous
`map_dataset_file_sync` wrapper on an authenticated deployment, call `raise_for_error()` on the
result: reading `.results` without checking `.error` silently accepts a truncated stream as a
complete one.

---

## Part 11: Failure handling

Two kinds of failure are worth distinguishing, and the harmonization report already separates them
for you.

An **errored** entity is one whose mapping call failed. That is "we do not know", which is a
different claim from "it did not resolve", so `HarmonizationResult` counts them apart:
`a_errors` / `b_errors` against `a_unresolved` / `b_unresolved`. Neither bucket is ever silently
dropped, which is the property that lets a run report a complete accounting of its inputs.

A **request** failure depends on its kind. An authentication failure (HTTP 401/403) raises
`BioMapperAuthError` from every entry point, because it is systemic. On the batch path used by
`resolve()` here, transient failures (429, 5xx, timeouts) do not raise: each affected entry comes
back with `.error` set. The single-entity call (`await client.map_entity(...)`) raises the typed
exceptions directly.

In [20]:
import biomapper.mapper as sync_api
from biomapper import BioMapperAuthError, BioMapperError


def safe_resolve(names):
    """Resolve, surfacing BOTH failure modes. Returns [] only when the call itself failed."""
    try:
        results = resolve(names)
    except BioMapperAuthError as exc:
        # Systemic: this deployment wants a key, or rejected the one sent. Not retryable.
        print(f"authentication failed: {exc}")
        return []
    except BioMapperError as exc:
        print(f"mapping failed: {exc}")
        return []
    # On the batch path a transient or per-entity failure is reported in `.error` rather than
    # raised, so the per-result check is not optional. Skipping it is how a failed entry gets
    # read as an unresolved one.
    for r in results:
        if r.error:
            print(f"entry-level failure for {r.query_name!r}: {r.error}")
    return results


# Never index the return value without checking it: on a caught exception it is empty, and an
# entry that came back carrying `.error` is not a result to read a CURIE off.
caffeine = safe_resolve(["Caffeine"])
if not caffeine:
    print("no results: the call failed and was handled above")
elif caffeine[0].error:
    print(f"Caffeine did not resolve cleanly: {caffeine[0].error}")
else:
    print(f"ok: Caffeine -> {caffeine[0].primary_curie}")

# Keyless by default (1.5.4): with no key configured, the plain synchronous wrapper works, with no
# anonymous flag and no client. The env var is cleared for the duration so this cell behaves
# identically for a reader who has a key set; otherwise their key would be sent and the output
# below would no longer demonstrate the keyless default.
_saved_key = os.environ.pop("BIOMAPPER_API_KEY", None)
try:
    keyless = sync_api.map_entity("Caffeine", base_url=API_ENDPOINT, timeout=API_TIMEOUT)
    # The sync wrapper uses the batch path, so a transient failure arrives in `.error`.
    if keyless.error:
        print(f"\nsync map_entity with no key configured failed: {keyless.error}")
    else:
        print(f"\nsync map_entity with no key configured: {keyless.query_name} -> "
              f"{keyless.primary_curie}")
finally:
    if _saved_key is not None:
        os.environ["BIOMAPPER_API_KEY"] = _saved_key


ok: Caffeine -> RM:0032992



sync map_entity with no key configured: Caffeine -> RM:0032992


---

## Part 12: Reproducing this notebook

Every output above came from one backend build, pinned in Part 1 and read live from `/health`. To
reproduce or to cite a run, record all of the following together:

| Field | Where it comes from |
|---|---|
| `biomapper` version | `biomapper.__version__` (installed distribution metadata) |
| resolution endpoint | the `base_url` passed to the client |
| `kestrel_version` | `GET {kestrel}/health` |
| `kg_version` **and** `kraken_package_version` | `GET {kestrel}/health`, both required, they diverge |
| `biolink_version`, `build_timestamp`, `git_commit` | `GET {kestrel}/health` |
| `source_versions` | `GET {kestrel}/health`, also the circularity register |
| input panel checksum | your own, over the exact file submitted |

Do not hardcode a build. Node assignments move, Part 6 is a worked example of a build change
altering a headline count, and a number without a pinned build cannot be audited or reproduced.

### Where to go next

- `biomapper.harmonize` is entirely local and offline. `harmonize`, `link_by_intersection`,
  `curie_set` and `predicted_curies` issue no requests, so they can be applied to results you
  already have on disk.
- `HarmonizationResult.summary()` is a counts-only dictionary, safe to serialize into a run
  manifest alongside the provenance block from Part 1.
- `biomapper.benchmarks` runs the external benchmark suite against a deployment, with the same
  provenance pinning, via `python -m biomapper.benchmarks all`.
- Issues and API requests: [github.com/Phenome-Health/biomapper/issues](https://github.com/Phenome-Health/biomapper/issues)